# Autoresearch-GLM Experiment Analysis

Analysis of autonomous feature-search results on the TaiwanCredit benchmark from `results.tsv`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Expected columns: commit, val_auc, status, description
df = pd.read_csv('results.tsv', sep='\t')
df['val_auc'] = pd.to_numeric(df['val_auc'], errors='coerce')
df['status'] = df['status'].astype(str).str.strip().str.upper()

if 'num_features' in df.columns:
    df['num_features'] = pd.to_numeric(df['num_features'], errors='coerce')

print(f'Total experiments: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head(10)


In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')
print(f'Crashes: {n_crash}')


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments ({len(kept)} total):\n')
for i, row in kept.iterrows():
    extra = ''
    if 'num_features' in row and pd.notna(row['num_features']):
        extra = f"  features={int(row['num_features'])}"
    print(f"  #{i:3d}  auc={row['val_auc']:.6f}{extra}  {row['description']}")


## Val AUC Over Time

Track how the best kept validation AUC evolves as experiments progress. Higher is better, so the frontier is the running maximum.


In [ ]:
def short_label(desc):
    desc = str(desc).strip().lower()
    if desc == 'baseline':
        return 'baseline'

    custom = {
        'collapse train.py to one explicit policy': 'one-policy',
        'try wider screen and larger interaction set': 'wider screen + inter',
        'try a deeper interaction shortlist': '+8 inter',
        'try an even deeper interaction shortlist': '+10 inter',
        'try a larger feature cap with more interactions': 'cap34',
        'try a still larger transformed feature cap': 'cap38',
        'try slightly stronger clipping': 'clip 98',
        'try stronger clipping again': 'clip 97',
        'try clipping at the 96th percentile': 'clip 96',
        'try light regularization with stronger clipping': 'clip 96 + l2',
    }
    if desc in custom:
        return custom[desc]

    if desc.startswith('try '):
        desc = desc[4:]
    replacements = [
        ('interaction', 'inter'),
        ('regularization', 'l2'),
        ('feature cap', 'cap'),
        ('screening', 'screen'),
        ('clipping', 'clip'),
        ('stronger ', ''),
        ('slightly ', ''),
        (' transformed ', ' xform '),
        (' transform set', ' transforms'),
        (' explicit ', ' '),
        (' policy', ''),
    ]
    for old, new in replacements:
        desc = desc.replace(old, new)
    desc = ' '.join(desc.split())
    if len(desc) > 24:
        desc = desc[:21].rstrip() + '...'
    return desc

fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
baseline_auc = valid.loc[0, 'val_auc']
kept = valid[valid['status'] == 'KEEP'].copy()
kept['running_best'] = kept['val_auc'].cummax()
kept['delta'] = kept['running_best'].diff().fillna(0.0)
kept['label'] = kept['description'].map(short_label)
best_auc = kept['running_best'].max()

focus_floor = baseline_auc - max(0.0007, (best_auc - baseline_auc) * 0.28)
focus = valid[valid['val_auc'] >= focus_floor].copy()
discarded = focus[focus['status'] == 'DISCARD']
kept_focus = focus[focus['status'] == 'KEEP']

milestone_idx = {kept.index[0], kept.index[-1]}
extra = kept.iloc[1:-1][kept.iloc[1:-1]['delta'] > 0].nlargest(min(3, max(len(kept) - 2, 0)), 'delta')
milestone_idx.update(extra.index.tolist())
milestones = kept.loc[sorted(milestone_idx)]

ax.scatter(discarded.index, discarded['val_auc'], c='#d4d4d4', s=12, alpha=0.55, zorder=2, label='Discarded')
ax.scatter(kept_focus.index, kept_focus['val_auc'], c='#2ecc71', s=50, zorder=4, label='Kept', edgecolors='black', linewidths=0.5)
ax.step(kept.index, kept['running_best'], where='post', color='#27ae60', linewidth=2.0, alpha=0.75, zorder=3, label='Running best')
ax.axhline(baseline_auc, color='#34495e', linestyle='--', linewidth=1.4, alpha=0.8, label=f'Baseline ({baseline_auc:.4f})')

for n, (idx, row) in enumerate(milestones.iterrows()):
    is_first = idx == kept.index[0]
    is_last = idx == kept.index[-1]
    if is_last:
        x_offset, y_offset, align = -8, 8, 'right'
    elif is_first:
        x_offset, y_offset, align = 5, 5, 'left'
    else:
        x_offset, y_offset, align = 6, 10 + 4 * (n % 2), 'left'
    ax.annotate(
        row['label'],
        (idx, row['val_auc']),
        xytext=(x_offset, y_offset),
        textcoords='offset points',
        ha=align,
        va='bottom',
        fontsize=8.2,
        rotation=18,
        color='#1f6f43',
        bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.78),
    )

n_total = len(df)
n_kept = len(df[df['status'] == 'KEEP'])
ax.set_xlabel('Experiment #', fontsize=12)
ax.set_ylabel('Validation AUC (higher is better)', fontsize=12)
ax.set_title(f'Autoresearch-GLM Progress: {n_total} Experiments, {n_kept} Kept Improvements', fontsize=14)
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, alpha=0.18)

margin = max(0.00035, (best_auc - focus_floor) * 0.08)
ax.set_xlim(-0.5, len(valid) - 0.5)
ax.set_ylim(focus_floor - margin, best_auc + margin)

plt.tight_layout()
plt.savefig('progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to progress.png')


## Summary Statistics


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
baseline_auc = df.iloc[0]['val_auc']
best_auc = kept['val_auc'].max()
best_row = kept.loc[kept['val_auc'].idxmax()]

print(f'Baseline val_auc:  {baseline_auc:.6f}')
print(f'Best val_auc:      {best_auc:.6f}')
print(f'Total improvement: {best_auc - baseline_auc:+.6f} ({(best_auc - baseline_auc) / baseline_auc * 100:.2f}%)')
print(f"Best experiment:   {best_row['description']}")
print()
print('Cumulative frontier improvements:')
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    print(f"  Experiment #{int(row['index']):3d}: auc={row['val_auc']:.6f}  {row['description']}")


## Top Hits (Kept Experiments by Improvement)


In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
kept['prev_auc'] = kept['val_auc'].shift(1)
kept['delta'] = kept['val_auc'] - kept['prev_auc']
hits = kept.iloc[1:].copy().sort_values('delta', ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'AUC':>10}  Description")
print('-' * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}  {row['val_auc']:.6f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")
